In [1]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

# 1. Setup Environment
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
os.environ["TRAINING_ENV"] = "kaggle"

# 2. Clone Repository
REPO_URL = "https://github.com/Firojpaudel/chatterbox-nepali.git"
REPO_DIR = "/kaggle/working/chatterbox-nepali"

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Updating repository...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)

# 3. Install Dependencies
print("Installing dependencies...")
# 💡 FIX: Explicitly install matched torch/torchvision versions to avoid 'torchvision::nms' errors on Kaggle
subprocess.run(["pip", "install", "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run(["pip", "install", "-e", "."], check=True)
subprocess.run(["pip", "install", "omegaconf", "conformer", "s3tokenizer", "safetensors", "pyloudnorm", "wandb", "huggingface-hub"], check=True)
subprocess.run(["pip", "install", "git+https://github.com/resemble-ai/Perth.git@master"], check=True)

print("\n--- Chatterbox Setup Complete ---")

In [2]:
%cd /kaggle/working/chatterbox-nepali

In [3]:
# Download existing checkpoint from Hugging Face
from huggingface_hub import hf_hub_download
print("Downloading t3_nepali_epoch_20.pt...")
ckpt_path = hf_hub_download(repo_id="officialuser/chatterbox-nepali", filename="t3_nepali_epoch_20.pt")
dest = "t3_nepali_epoch_20.pt"
if os.path.exists(dest): os.remove(dest)
os.symlink(ckpt_path, dest)
print(f"✅ Checkpoint ready at {dest}")

In [4]:
!ls finetuning_data/manifests/

In [5]:
# Download manifests and audio from your Hugging Face dataset
import os
from pathlib import Path
from huggingface_hub import snapshot_download

dataset_repo = "Firoj112/voxcpm-nepali-data"
local_dir = "finetuning_data"

print(f"Downloading dataset from {dataset_repo}...")
# Download everything (manifests and wavs)
snapshot_download(
    repo_id=dataset_repo, 
    repo_type="dataset", 
    local_dir=local_dir,
    token=os.environ.get("HF_TOKEN")
)

print(f"✅ Dataset ready in {local_dir}")

# Quick check
wav_count = len(list(Path(local_dir).glob("wavs/*.wav")))
manifest_count = len(list(Path(local_dir).glob("manifests/*.jsonl")))
print(f"Found {wav_count} WAV files and {manifest_count} manifests.")

In [6]:
# Verify manifests exist and show line counts
from pathlib import Path
m = Path("finetuning_data/manifests")
if not m.exists():
    print("Manifest directory missing")
else:
    print("Manifests present:", [p.name for p in m.glob("*.jsonl")])
    for p in m.glob("*.jsonl"):
        size = p.stat().st_size
        with open(p, encoding='utf-8') as f:
            lines = sum(1 for _ in f)
        print(f"{p.name}: {lines} lines, {size} bytes")

In [7]:
# 🚀 START DISTRIBUTED TRAINING (T4 x 2)
# We use torchrun for distributed training across 2 GPUs
!torchrun --nproc_per_node=2 src/chatterbox/train_nepali.py \
    --manifest finetuning_data/manifests/train_manifest.jsonl \
    --resume_t3_weights t3_nepali_epoch_20.pt \
    --batch_size 16 \
    --epochs 50 \
    --lr 2e-5 \
    --save_every 5 \
    --num_workers 2 \
    --distributed

In [ ]:
# 📤 PUSH FINAL MODEL TO HUGGING FACE
from huggingface_hub import HfApi
import os

api = HfApi()
repo_id = "officialuser/chatterbox-nepali"
token = os.environ.get("HF_TOKEN")

if token and os.path.exists("t3_mtl_nepali_final.safetensors"):
    print(f"Pushing model to {repo_id}...")
    api.upload_file(
        path_or_fileobj="t3_mtl_nepali_final.safetensors",
        path_in_repo="t3_mtl_nepali_final.safetensors",
        repo_id=repo_id,
        repo_type="model",
        token=token
    )
    print("✅ Successfully pushed to Hugging Face!")
else:
    print("❌ HF_TOKEN missing or final model not found.")